# Start here: what we are building, in 8 small steps

**How to use this:** click a grey code box and press **Shift + Enter** to run it. Go top to
bottom. Every code box is one or two lines; nothing can break the data (it's read-only).

### What are we building?

A **helper you can ask business questions in plain English**, like *"What is our late-delivery
rate?"*, that finds the answer in your database and shows you how it got it.

### Is it training an AI? Is it RAG?

**No training.** We use a ready-made AI (Claude or Gemini). And **not RAG**: RAG means
"search documents, paste them into the AI". Our answers live in a *database*, so instead we
let the AI **ask our code to look things up**, and our code does the looking safely.

Think of it like this:

> **The AI is a smart new analyst** who doesn't know your company yet.
> **Our code is the assistant** who holds the keys to the database.
> The analyst can only *ask* the assistant: "show me the tables", "run this query".
> The assistant checks every request is safe, runs it, and hands back the result.

In [21]:
from tour import *   # loads the project (one-time setup)

## Step 1. Look at the data

This is the Olist dataset: a real Brazilian online shop, about 100,000 orders.

In [3]:
tables()

,table,rows,columns
0,customers,99441,5
1,geolocation,1000163,5
2,leads_closed,842,14
3,leads_qualified,8000,4
4,order_items,112650,7
5,order_payments,103886,5
6,order_reviews,99224,7
7,orders,99441,8
8,product_category_name_translation,71,2
9,products,32951,9


## Step 2. Ask the database a question yourself (in SQL)

This is exactly what the AI will do later. Change the question and run it again.

In [8]:
run_sql("SELECT order_status, COUNT(*) AS orders FROM orders GROUP BY order_status")

,order_status,orders
0,approved,2
1,canceled,625
2,created,5
3,delivered,96478
4,invoiced,314
5,processing,301
6,shipped,1107
7,unavailable,609


## Step 3. Try to break it

Our code has a **safety guard**: it only lets *reading* through, never changing or deleting.

In [7]:
run_sql("DELETE FROM orders")

'BLOCKED by the safety guard: only SELECT queries are allowed, got DELETE'

**Try it:** put your own sneaky query between the quotes, e.g. `DROP TABLE orders`.

## Step 4. What the AI is told about the data

Before the AI writes any SQL, our code studies each table: how many values are empty, how many
are different, some examples. Look at the two ID columns below:

In [16]:
what_the_ai_knows("customers")

,column,empty values,different values,every value unique?,examples
0,customer_id,0%,99441,yes,"06b8999e2fba1a1fbc88172c00ba8bc7, 18955e83d337..."
1,customer_unique_id,0%,96096,no,"861eff4711a542e4b93843c6dd7febb0, 290c77bc529b..."
2,customer_zip_code_prefix,0%,14994,no,"14409, 9790, 1151"
3,customer_city,0%,4119,no,"franca, sao bernardo do campo, sao paulo"
4,customer_state,0%,27,no,"SP, SC, MG"


`customer_id` is **unique for every row** but `customer_unique_id` is **not**. Why? Olist makes
a new `customer_id` for every *order*. So counting `customer_id` counts orders (99,441), while
counting `customer_unique_id` counts real people (96,096). Without this table the AI would
easily get it wrong.

Our code also works out **how the tables connect**:

In [17]:
joins()

,this column,points to
0,leads_closed.mql_id,leads_qualified.mql_id
1,leads_closed.seller_id,sellers.seller_id
2,order_items.order_id,orders.order_id
3,order_items.product_id,products.product_id
4,order_items.seller_id,sellers.seller_id
5,order_payments.order_id,orders.order_id
6,order_reviews.order_id,orders.order_id
7,orders.customer_id,customers.customer_id
8,products.product_category_name,product_category_name_translation.product_cate...


## Step 5. The business definitions you approved

"Revenue" could mean several things. You chose the meaning; the AI must follow it.

In [7]:
definitions()

,term,means
0,valid_order,An order whose status is not 'canceled' or 'un...
1,revenue,Sum of item price over valid orders. Excludes ...
2,order_date,orders.order_purchase_timestamp. Months are ca...
3,aov,Average order value = revenue / number of dist...
4,delivered_order,order_status = 'delivered' and order_delivered...
5,late_delivery,A delivered order whose delivery day is after ...
6,delivery_days,Days from purchase to delivery to the customer.
7,customer,"A person, identified by customers.customer_uni..."
8,repeat_customer,A customer with 2 or more distinct valid orders.
9,category,English name from product_category_name_transl...


## Step 6. Watch the AI work, step by step

Here a **pretend AI** follows a script, so you can see each step clearly and for free.
Everything else (the safety guard, the database, the checking) is the real thing.

In [22]:
answer = watch_the_ai()

QUESTION: What is our late-delivery rate?

Step 1
   AI asks: Which tables are there?
   OUR CODE answers: [{"table": "customers", "rows": 99441, "columns": 5}, {"table": "geolocation", "rows": 1000163, "columns": 5},...

Step 2
   AI asks: What's in the orders table?
   OUR CODE answers: {"table": "orders", "rows": 99441, "columns": [{"name": "order_id", "type": "TEXT", "null_share": 0.0, "distin...

Step 3
   AI asks: Please run this SQL: SELECT order_status, COUNT(*) FROM orders GROUP BY 1
   OUR CODE answers: {"columns": ["order_status", "COUNT(*)"], "rows": [["approved", 2], ["canceled", 625], ["created", 5], ["deliv...

Step 4
   AI asks: I'm done. My answer: 6.77% of delivered orders arrived late.

FINAL ANSWER: 6.77% of delivered orders arrived late.
Our code re-ran the AI's final SQL to double-check it and got: [[6.77]]


That's the whole idea: **the AI asks, our code checks and answers, repeat**, and at the end our
code re-runs the AI's final SQL itself, so every answer is backed by a real query.

**Try it:** make the pretend AI try something dangerous in its third step:
`watch_the_ai(sql_to_try="DELETE FROM orders")`. The guard blocks it and the AI is told why.

## Step 7. Check the answer against the known-correct one

We keep 30 questions with answers we already know (the "golden set"). That's how we'll measure
the real AI later.

In [9]:
check(answer)

'CORRECT: the AI said 6.77, the known answer is 6.77'

## Step 8. What the real AI receives (a peek)

When you ask a real question, the AI first gets instructions like these, plus the table list and
your definitions:

In [10]:
system_prompt_preview()

You are a data analyst answering business questions about one database. Work from the data:
look at the schema, run queries to check values and joins, and base every number you report
on a query you ran. When you are done, call submit_answer with a short answer and the one
SELECT that produces it.

Database: olist (sqlite SQL). Queries are read-only and row-limited.

Tables:
- customers (99441 rows): customer_id TEXT, customer_unique_id TEXT, customer_zip_code_prefix INTEGER, customer_city TEXT, customer_state TEXT
- geolocation (1000163 rows): geolocation_zip_code_prefix INTEGER, geolocation_lat REAL, geolocation_lng REAL, geolocation_city TEXT, geolocation_state TEXT
- leads_closed (842 rows): mql_id TEXT, seller_id TEXT, sdr_id TEXT, sr_id TEXT, won_date TEXT, business_segment TEXT, lead_type TEXT, lead_behaviour_profile TEXT, has_company INTEGER, has_gtin INTEGER, average_stock TEXT, 
...


## Recap

| Piece | Job | Like... |
|---|---|---|
| Connector | talks to the database, read-only | the assistant with the keys |
| Safety guard | lets only safe reading queries through | security at the door |
| Catalog | studies the tables and their connections | notes for the new analyst |
| Definitions | what words like "revenue" mean here | the company glossary |
| Agent loop | AI asks, our code answers, repeat | the conversation |
| Golden set | questions with known answers | the exam |

When this feels familiar, the detailed version is **`01_how_it_works.ipynb`**.